# 芯片测试 1/3：解析 ATE 日志、数据体检与清洗

**目标**：把测试机写出的 STDF 风格文本日志（百万行级）流式解析成一张"一行一颗芯片"的干净宽表，
并且在分析之前先交代清楚：丢了什么、修了什么、哪些数字不能信。

**输入**：`data/raw/chip/*.log`（模拟日志，由 `scripts/generate_chip_data.py` 生成，全部是模拟数据）

**输出**：
- `data/processed/chip_parts.csv`：清洗后的芯片表（一条测试记录一行，带首测/复测标注）
- `data/processed/chip_tests.csv`：测试项目录（单位、上下限、耗时）

**下一步**：Notebook 02 读取这两个文件做良率归因。

这个 Notebook 与 Python 版（`chiptest/logparse.py`、`chiptest/data.py`）是同一套逻辑，
两个输出文件与 Python 版逐字节一致，由 `tests/test_chip_notebook_parity.py` 守住。

In [1]:
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd()
ROOT = next((folder.parent for folder in (cwd, *cwd.parents) if folder.name == "notebooks"), cwd)
RAW = ROOT / "data" / "raw" / "chip"
PROCESSED = ROOT / "data" / "processed"
CLEAN = PROCESSED / "chip_parts.csv"
TESTS = PROCESSED / "chip_tests.csv"

## 1. 先看日志长什么样

一个文件对应一个 lot。文件头是 bin 目录（`HBR`）和测试项目录（`TSR`），之后每片晶圆一段：
`WIR` 开始 → 每颗芯片 `PIR` + 一组 `PTR`（每个测试项一条读数）+ `PRR`（判定）→ `WRR` 结束。

In [2]:
log_files = sorted(RAW.glob("*.log"))
assert log_files, "找不到日志：先运行 `uv run python scripts/generate_chip_data.py`"
total_mb = sum(p.stat().st_size for p in log_files) / 1e6
print(f"{len(log_files)} 个日志文件，共 {total_mb:.1f} MB\n")

lines = log_files[0].read_text(encoding="ascii", errors="replace").splitlines()
print("\n".join(lines[:35]))

8 个日志文件，共 21.5 MB

# DEMO-SOC SYNTHETIC ATE DATALOG v1 (all data simulated)
HBR|1|PASS
HBR|2|CONTACT
HBR|3|LEAKAGE
HBR|4|POWER
HBR|5|ANALOG
HBR|6|SPEED
HBR|7|MEMORY
HBR|8|FUNCTIONAL
TSR|1001|CONT_VDD|ohm|0|2|20
TSR|1002|CONT_IO|ohm|0|2|20
TSR|1101|IDD_STANDBY|uA|0|150|60
TSR|1102|IDD_ACTIVE|mA|60|200|80
TSR|1201|VREF|V|1.17|1.23|50
TSR|1202|LDO_OUT|V|1.755|1.845|50
TSR|1203|OSC_FREQ|MHz|23.4|24.6|40
TSR|1204|PLL_LOCK|us|0|50|60
TSR|1205|ADC_OFFSET|LSB|-3|3|120
TSR|1301|SCAN_FMAX|MHz|1750|-|400
TSR|1302|VMIN|V|-|0.78|300
TSR|1401|SRAM_BIST|cnt|0|0|800
TSR|1402|FUNC_PAT|cnt|0|0|350
WIR|L01|01|T01|PROG_v1.3|1|2026-10-02T08:00:00
PIR|0001|0|-17|3|2026-10-02T08:00:00
PTR|1001|0.41012|P
PTR|1002|0.40021|P
PTR|1101|32.759|P
PTR|1102|111.6|P
PTR|1201|1.1926|P
PTR|1202|1.7896|P
PTR|1203|24.003|P
PTR|1204|15.433|P
PTR|1205|-1.8276|P
PTR|1301|1972.3|P
PTR|1302|0.72281|P


In [3]:
kinds = Counter(line.split("|", 1)[0] for line in lines)
pd.Series(kinds, name="lines").to_frame()

,lines
# DEMO-SOC SYNTHETIC ATE DATALOG v1 (all data simulated),1
HBR,8
TSR,260
WIR,20
PIR,9514
PTR,123682
PRR,9494
WRR,20


| 记录 | 含义 | 字段 |
|---|---|---|
| `HBR` | bin 目录 | bin 号 \| 名称 |
| `TSR` | 测试项声明 | 测试号 \| 名称 \| 单位 \| 下限 \| 上限 \| 耗时(ms)，`-` 表示没有该侧限值 |
| `WIR` | 一片晶圆的一次测试开始 | lot \| 晶圆号 \| 测试机 \| 程序 \| 第几次测试 \| 时间 |
| `PIR` | 一颗芯片开始 | 芯片号 \| x \| y \| site \| 时间 |
| `PTR` | 一个测试项的读数 | 测试号 \| 数值 \| 标志 |
| `PRR` | 一颗芯片结束 | 芯片号 \| 硬件 bin \| P/F |
| `WRR` | 晶圆测试结束 | lot \| 晶圆号 \| 第几次测试 \| 芯片数 |

`PTR` 占了绝大多数行，是解析的热路径。

## 2. 流式解析器：设计取舍

- **逐行读、按晶圆整理**：解析时保留已完成的 DataFrame，收齐后再合并和导出。输入按行读取，但内存会随保留的记录增加。GB 级日志可考虑分批写盘和分区处理。
- **热路径不用正则**：`PTR` 只做 `split` + 一次字典查找 + `float()`，不逐行建 DataFrame。
- **常见异常会计数**：截断的参数记录、哨兵值、nan、缺失 `PRR`、重复芯片块等由体检报告交代。
- **单位以 `TSR` 声明为准**：同一个测试项在不同晶圆可能换单位（`uA` 和 `mA`），
  统一换算成第一次见到的单位；换算不了的（量纲对不上）宁可丢掉也不混进去。

In [4]:
SENTINEL = 1e30
SI_PREFIX = {"p": 1e-12, "n": 1e-9, "u": 1e-6, "m": 1e-3, "k": 1e3, "M": 1e6, "G": 1e9}
BASE_UNITS = {"A", "V", "Hz", "s", "ohm", "LSB", "cnt"}


def split_unit(unit):
    if unit in BASE_UNITS:
        return 1.0, unit
    if len(unit) > 1 and unit[0] in SI_PREFIX and unit[1:] in BASE_UNITS:
        return SI_PREFIX[unit[0]], unit[1:]
    return None


def si_factor(unit_from, unit_to):
    """把 unit_from 的数值换算到 unit_to 要乘的系数；量纲不同或不认识的单位返回 None。"""
    a, b = split_unit(unit_from), split_unit(unit_to)
    if a is None or b is None or a[1] != b[1]:
        return None
    return float(f"{a[0] / b[0]:.10g}")


for a, b in [("mA", "uA"), ("uA", "uA"), ("uA", "mA"), ("V", "uA"), ("foo", "V")]:
    print(f"{a:>4} -> {b:<3} 乘 {si_factor(a, b)}")

  mA -> uA  乘 1000.0
  uA -> uA  乘 1.0
  uA -> mA  乘 0.001
   V -> uA  乘 None
 foo -> V   乘 None


In [5]:
def limit(text):
    return float("nan") if text == "-" else float(text)


def parse_logs(paths):
    stats = Counter()
    tests = {}  # 测试号 -> 声明信息，保持首次出现的顺序
    bins = {}
    col_of = {}  # PTR 里的测试号文本 -> 值向量的列号
    factor = []  # 每列的单位换算系数：1.0 不用换，nan 表示单位对不上
    frames = []

    def flush(wafer, rows):
        if not rows:
            return
        lot, wafer_id, pass_no, station, program = wafer
        part_id, x, y, site, ts, hbin, ok, values = zip(*rows.values(), strict=True)
        names = [t["name"] for t in tests.values()]
        matrix = np.full((len(rows), len(names)), np.nan)
        for i, v in enumerate(values):
            matrix[i, : len(v)] = v
        meta = pd.DataFrame(
            {
                "lot_id": lot,
                "wafer_id": wafer_id,
                "pass_no": pass_no,
                "part_id": part_id,
                "x": np.array(x),
                "y": np.array(y),
                "site": np.array(site),
                "station": station,
                "program": program,
                "test_time": ts,
                "hbin": np.array(hbin),
                "tester_pass": np.array(ok),
            }
        )
        frames.append(pd.concat([meta, pd.DataFrame(matrix, columns=names)], axis=1))
        stats["parts"] += len(rows)

    for path in sorted(paths):
        stats["files"] += 1
        wafer, rows, part = None, {}, None  # part = [芯片号, x, y, site, 时间, 读数向量]
        with path.open(encoding="ascii", errors="replace") as fh:
            for line in fh:
                stats["lines"] += 1
                rec = line.rstrip("\n").split("|")
                kind = rec[0]

                if kind == "PTR":
                    stats["ptr_lines"] += 1
                    j = col_of.get(rec[1]) if len(rec) == 4 and part is not None else None
                    if j is None:
                        stats["bad_lines"] += 1
                        continue
                    try:
                        v = float(rec[2])
                    except ValueError:
                        stats["bad_lines"] += 1
                        continue
                    if v != v:
                        stats["nan_values"] += 1
                    elif abs(v) >= SENTINEL:
                        stats["sentinel_values"] += 1
                    else:
                        f = factor[j]
                        if f != 1.0:
                            if f != f:
                                stats["unknown_unit_values"] += 1
                                continue
                            v *= f
                            stats["unit_converted_values"] += 1
                        part[5][j] = v
                elif kind == "PIR":
                    if part is not None:
                        stats["incomplete_parts"] += 1
                    try:
                        part = [
                            rec[1],
                            int(rec[2]),
                            int(rec[3]),
                            int(rec[4]),
                            rec[5],
                            [np.nan] * len(factor),
                        ]
                    except (IndexError, ValueError):
                        stats["bad_lines"] += 1
                        part = None
                elif kind == "PRR":
                    if part is None or len(rec) != 4 or rec[1] != part[0]:
                        stats["bad_lines"] += 1
                        continue
                    if rec[1] in rows:
                        stats["duplicate_parts"] += 1
                    else:
                        rows[rec[1]] = (*part[:5], int(rec[2]), rec[3] == "P", part[5])
                    part = None
                elif kind == "TSR":
                    no, name, unit = int(rec[1]), rec[2], rec[3]
                    lo, hi, ms = limit(rec[4]), limit(rec[5]), float(rec[6])
                    known = tests.get(no)
                    if known is None:
                        tests[no] = {
                            "test_no": no,
                            "name": name,
                            "unit": unit,
                            "lo": lo,
                            "hi": hi,
                            "time_ms": ms,
                        }
                        col_of[rec[1]] = len(factor)
                        factor.append(1.0)
                    else:
                        f = si_factor(unit, known["unit"])
                        factor[col_of[rec[1]]] = np.nan if f is None else f
                elif kind == "WIR":
                    wafer = (rec[1], rec[2], int(rec[5]), rec[3], rec[4])
                    rows, part = {}, None
                elif kind == "WRR":
                    if part is not None:
                        stats["incomplete_parts"] += 1
                        part = None
                    if wafer is not None:
                        stats["wafer_passes"] += 1
                        stats["wafer_count_mismatch"] += len(rows) != int(rec[4])
                        flush(wafer, rows)
                    wafer, rows = None, {}
                elif kind == "HBR":
                    bins[int(rec[1])] = rec[2]
                elif kind and kind[0] != "#":
                    stats["bad_lines"] += 1

    parts = pd.concat(frames, ignore_index=True)
    parts["test_time"] = pd.to_datetime(
        parts["test_time"], format="%Y-%m-%dT%H:%M:%S", errors="coerce"
    )
    return parts, pd.DataFrame(list(tests.values())), bins, stats

In [6]:
start = time.perf_counter()
parts, tests, bins, stats = parse_logs(log_files)
elapsed = time.perf_counter() - start
print(
    f"{stats['lines']:,} 行 / {total_mb:.1f} MB，用时 {elapsed:.2f}s"
    f"（{stats['lines'] / elapsed / 1e6:.2f}M 行/秒）"
)
print(f"解析出 {len(parts):,} 条芯片记录 x {len(tests)} 个测试项\n")
display(tests)

1,165,340 行 / 21.5 MB，用时 0.60s（1.95M 行/秒）
解析出 77,265 条芯片记录 x 13 个测试项



,test_no,name,unit,lo,hi,time_ms
0,1001,CONT_VDD,ohm,0.000,2.000,20.0
1,1002,CONT_IO,ohm,0.000,2.000,20.0
2,1101,IDD_STANDBY,uA,0.000,150.000,60.0
3,1102,IDD_ACTIVE,mA,60.000,200.000,80.0
4,1201,VREF,V,1.170,1.230,50.0
5,1202,LDO_OUT,V,1.755,1.845,50.0
6,1203,OSC_FREQ,MHz,23.400,24.600,40.0
7,1204,PLL_LOCK,us,0.000,50.000,60.0
8,1205,ADC_OFFSET,LSB,-3.000,3.000,120.0
9,1301,SCAN_FMAX,MHz,1750.000,NaN,400.0


### 脏数据长什么样

日志里混有四类问题，解析器都按"计数 + 丢弃"处理。下面各取第一个文件里的一个例子：

In [7]:
EXPECTED_FIELDS = {"PIR": 6, "PTR": 4, "PRR": 4}
examples = {}
for line in lines:
    rec = line.split("|")
    if rec[0] in EXPECTED_FIELDS and len(rec) != EXPECTED_FIELDS[rec[0]]:
        examples.setdefault("截断的行（字段数不对）", line)
    elif rec[0] == "PTR" and rec[2] == "1.0E+38":
        examples.setdefault("哨兵值（测试机用极大值表示没测）", line)
    elif rec[0] == "PTR" and rec[2] == "nan":
        examples.setdefault("nan", line)
for label, line in examples.items():
    print(f"{label}: {line}")

nan: PTR|1204|nan|P
哨兵值（测试机用极大值表示没测）: PTR|1302|1.0E+38|P
截断的行（字段数不对）: PTR|1202|1.


另外两类不在单行上看得出来：芯片块缺 `PRR`（判定丢了，整颗芯片丢弃），芯片块整段重复（只保留第一份）。
它们的数量在后面的体检报告里。

## 3. 清洗：芯片身份、首测/末次、按限值重算失败

一颗物理芯片 = `(lot, wafer, x, y)`。同一颗芯片可能有首测（pass 1）和复测（pass 2）两条记录，所以：

- **首测良率（FPY）**以首测为单位，同时反映芯片和测试系统；
- **最终良率**用最后一次测试的判定，是出货口径；
- 两者之差是"复测挽回"的部分。它可能是真的间歇性问题，也可能在掩盖测试硬件的缺陷。

另外不直接信测试机的 P/F：按 `TSR` 里的限值重新算每个测试项是否失败，再与测试机判定交叉核对。

In [8]:
DIE_KEY = ["lot_id", "wafer_id", "x", "y"]

df = parts.sort_values(["lot_id", "wafer_id", "pass_no", "part_id"]).reset_index(drop=True)

die = df.groupby(DIE_KEY)["pass_no"]
df["is_first_pass"] = df["pass_no"] == 1
df["is_final_pass"] = df["pass_no"] == die.transform("max")
df["retested"] = die.transform("max") > 1
final_ok = df.loc[df["is_final_pass"], [*DIE_KEY, "tester_pass"]].drop_duplicates(DIE_KEY)
df = df.merge(final_ok.rename(columns={"tester_pass": "final_ok"}), on=DIE_KEY, how="left")

df["date"] = df["test_time"].dt.normalize()
df["hbin_name"] = df["hbin"].map(bins)

n_first = int(df["is_first_pass"].sum())
print(f"{len(df):,} 条测试记录 = {n_first:,} 条首测 + {len(df) - n_first:,} 条复测")
print(
    f"{df.groupby(DIE_KEY).ngroups:,} 颗物理芯片，其中 {int(df.loc[df['is_first_pass'], 'retested'].sum()):,} 颗被复测"
)

77,265 条测试记录 = 71,907 条首测 + 5,358 条复测
71,917 颗物理芯片，其中 5,348 颗被复测


In [9]:
limits = tests.set_index("name")
lo, hi = limits["lo"], limits["hi"]
fails = {f"fail_{name}": (df[name] < lo[name]) | (df[name] > hi[name]) for name in limits.index}
fail_frame = pd.DataFrame(fails)
df = pd.concat([df, fail_frame], axis=1)
df["n_fail_tests"] = fail_frame.sum(axis=1)

complete = ~df[list(limits.index)].isna().any(axis=1)
df["flag_mismatch"] = complete & (df["tester_pass"] != (df["n_fail_tests"] == 0))
df.head(3).T.head(30)

,0,1,2
lot_id,L01,L01,L01
wafer_id,01,01,01
pass_no,1,1,1
part_id,0001,0002,0003
x,0,-5,-4
y,-17,-16,-16
site,3,2,1
station,T01,T01,T01
program,PROG_v1.3,PROG_v1.3,PROG_v1.3
test_time,2026-10-02 08:00:00,2026-10-02 08:00:00,2026-10-02 08:00:00


有缺失读数的芯片（哨兵值、nan 被丢掉）无法复核，不计入"判定不一致"。

## 4. 体检报告

先交代丢了什么、修了什么，再谈分析。这些数字也是向别人证明"这份数据可以用"的依据。

In [10]:
quality = {
    "files": stats["files"],
    "lines": stats["lines"],
    "PTR records": stats["ptr_lines"],
    "unparseable lines (truncated/orphan)": stats["bad_lines"],
    "sentinel values (1E+38, treated as missing)": stats["sentinel_values"],
    "nan values": stats["nan_values"],
    "values unit-converted (e.g. mA -> uA)": stats["unit_converted_values"],
    "wafer passes": stats["wafer_passes"],
    "wafer passes whose part count != WRR": stats["wafer_count_mismatch"],
    "parts parsed": stats["parts"],
    "incomplete parts dropped (no PRR)": stats["incomplete_parts"],
    "duplicate part blocks dropped": stats["duplicate_parts"],
}
test_cols = [c for c in tests["name"] if c in df.columns]
quality["parts with a missing test value"] = int(df[test_cols].isna().any(axis=1).sum())
quality["dies without a first pass (PRR lost)"] = int(
    df.groupby(DIE_KEY)["is_first_pass"].any().eq(False).sum()
)
quality["tester flag != recomputed from limits"] = int(df["flag_mismatch"].sum())
pd.Series(quality, name="count").to_frame()

,count
files,8
lines,1165340
PTR records,1007981
unparseable lines (truncated/orphan),306
"sentinel values (1E+38, treated as missing)",207
nan values,88
values unit-converted (e.g. mA -> uA),946
wafer passes,160
wafer passes whose part count != WRR,86
parts parsed,77265


读法：

- **丢弃的比例很小**，且都有明确归属（截断、哨兵、缺 `PRR`、重复），不是"悄悄丢的"。
- **单位换算**：有一部分晶圆把 `IDD_STANDBY` 写成了 `mA`，不换算的话这些读数会缩小一千倍，
  漏电异常的芯片会被悄悄放过，而且在图上看不出来。
- **"测试机判定 != 按限值重算"** 应当接近 0；非 0 才说明测试程序的限值和日志里声明的不一致，需要追查。

再做一个独立的交叉核对：`WRR` 记录里声明了每片晶圆应有多少颗芯片，把它们加起来，
减去实际解析出的芯片数，应该正好等于"缺 `PRR` 被丢弃"的芯片数。两份互不依赖的计数对得上，才说明没有多丢或漏丢。

In [11]:
declared = sum(
    int(line.split("|")[4])
    for path in log_files
    for line in path.read_text(encoding="ascii", errors="replace").splitlines()
    if line.startswith("WRR|")
)
missing = declared - stats["parts"]
print(f"WRR 声明的芯片总数 {declared:,}，解析出 {stats['parts']:,}，差 {missing}")
print(f"体检里'缺 PRR 丢弃'的芯片数 {stats['incomplete_parts']}")
print("对得上" if missing == stats["incomplete_parts"] else "对不上，需要追查")

WRR 声明的芯片总数 77,452，解析出 77,265，差 187
体检里'缺 PRR 丢弃'的芯片数 187
对得上


## 5. 保存

In [12]:
PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN, index=False)
tests.to_csv(TESTS, index=False)
print(f"saved {len(df):,} rows x {df.shape[1]} columns -> {CLEAN.relative_to(ROOT)}")
print(f"saved {len(tests)} tests -> {TESTS.relative_to(ROOT)}")

saved 77,265 rows x 46 columns -> data/processed/chip_parts.csv
saved 13 tests -> data/processed/chip_tests.csv


## 局限性

- 日志是**模拟的 STDF 风格文本**，不是二进制 STDF V4。真实环境先用 `pystdf` 或厂商工具转成这类结构，
  或者直接把这个解析器的状态机换成对应的二进制读取。
- 这里最后用 `pd.concat` 把所有晶圆拼回内存里一张表，对百万行没问题；日志大到内存放不下时，
  应改成每片晶圆直接写 Parquet 分片，后续分析按分片读取。
- 单位换算只认识 SI 前缀加一个白名单基本单位；不认识的会被丢弃并计数，不会静默混入。
- 脏数据比例、单位切换的晶圆都是按教学需要设定的，真实产线的脏数据形态要比这里杂得多。